# Reranking Strategies for Hybrid Search

## Purpose

This notebook compares four post-retrieval reranking strategies applied to hybrid search results. The current system uses the search pipeline for score combination (min_max normalization + arithmetic_mean with weights [0.3 BM25, 0.7 neural]) but performs **no reranking** after results are returned.

Reranking can improve result quality by re-scoring retrieved documents with a more sophisticated model that jointly considers the query and each passage.

## Strategies Compared

| # | Strategy | Model Required | Latency | Cost |
|---|----------|---------------|---------|------|
| 1 | No reranking (baseline) | None | Lowest | Free |
| 2 | Reciprocal Rank Fusion (RRF) | None | Low | Free |
| 3 | Cross-encoder reranking | `cross-encoder/ms-marco-MiniLM-L-6-v2` (22M params, ~80MB) | Medium | Free (local) |
| 4 | LLM-based reranking | Bedrock Claude 3 Haiku | High | ~$0.01/run |

## Prerequisites

- FastAPI server running on `localhost:8000`
- FAISS+BM25 search index populated with sagemaker-docs (336 documents)
- `pip install -r requirements.txt`
- AWS credentials configured (for LLM reranking via Bedrock)

In [ ]:
import sys
import time
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, ".")
from helpers import search, results_to_dataframe, hits_to_doc_ids, BASE_URL

# Verify the API is reachable
resp = requests.get(f"{BASE_URL}/search/index/stats", timeout=10)
stats = resp.json()
print(f"Index: {stats['index_name']} | Documents: {stats['doc_count']} | Status: {stats['status']}")

In [ ]:
# Test queries — using the vector-excelling and hybrid-excelling queries
# These are the most interesting for reranking because they mix concepts and terms
TEST_QUERIES = [
    "How do I make sure my notebook isn't exposed to the internet?",
    "What is the benefit of using a project instead of running pipelines directly?",
    "How can data scientists share code consistently across a team?",
    "What IAM permissions does an execution role need to run a training job?",
    "How does EventBridge trigger actions when an endpoint changes status?",
    "What kubectl commands do I use to check a training job running in Kubernetes?",
]

print(f"Test queries: {len(TEST_QUERIES)}")

---
## Strategy 1: No Reranking (Baseline)

The current production behavior. The search pipeline combines BM25 and neural scores:

1. **Normalization**: `min_max` — scales each score set to [0, 1]
2. **Combination**: `arithmetic_mean` with weights `[0.3, 0.7]` — 30% BM25 + 70% neural

This is a simple, fast approach but may not produce optimal ranking because:
- The weights are static (not query-dependent)
- Linear combination may not capture complex relevance signals
- No cross-attention between query and passage

In [ ]:
# Baseline: hybrid search with no reranking
baseline_results = {}
baseline_times = {}

for query in TEST_QUERIES:
    start = time.time()
    resp = search(query, search_type="hybrid", size=10)
    elapsed = time.time() - start
    baseline_results[query] = resp
    baseline_times[query] = elapsed
    print(f"  [{elapsed:.2f}s] {query[:60]}")

print(f"\nAverage latency: {np.mean(list(baseline_times.values())):.3f}s")

---
## Strategy 2: Reciprocal Rank Fusion (RRF)

**How it works:** Instead of letting the search pipeline combine BM25 and neural scores internally, we retrieve results from each search type separately and fuse them using RRF:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

Where $R$ is the set of result lists (text + vector), $k$ is a constant (typically 60), and $\text{rank}_r(d)$ is the 1-based rank of document $d$ in list $r$.

**Pros:**
- No model required — purely algorithmic
- Fast — just arithmetic over ranks
- Well-studied in information retrieval literature
- Robust to score scale differences between rankers

**Cons:**
- Ignores actual relevance scores — only uses rank positions
- The $k$ parameter is a hyperparameter (though 60 is a solid default)
- Cannot distinguish between "barely ranked" and "strongly ranked" at the same position

In [ ]:
def reciprocal_rank_fusion(
    result_lists: list[list[dict]],
    k: int = 60,
) -> list[dict]:
    """Fuse multiple ranked result lists using Reciprocal Rank Fusion.

    Args:
        result_lists: List of result lists, each containing hit dicts.
        k: RRF constant (default 60).

    Returns:
        Fused and re-sorted result list with added 'rrf_score' field.
    """
    scores: dict[str, float] = {}
    doc_data: dict[str, dict] = {}

    for results in result_lists:
        for rank, hit in enumerate(results, start=1):
            doc_id = hit["doc_id"]
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            if doc_id not in doc_data:
                doc_data[doc_id] = hit

    fused = []
    for doc_id, rrf_score in sorted(scores.items(), key=lambda x: -x[1]):
        entry = dict(doc_data[doc_id])
        entry["rrf_score"] = round(rrf_score, 6)
        fused.append(entry)
    return fused


print("RRF function defined.")

In [ ]:
# RRF: retrieve text and vector results separately, then fuse
rrf_results = {}
rrf_times = {}

for query in TEST_QUERIES:
    start = time.time()
    text_resp = search(query, search_type="text", size=20)
    vector_resp = search(query, search_type="vector", size=20)
    fused = reciprocal_rank_fusion(
        [text_resp["hits"], vector_resp["hits"]],
        k=60,
    )
    elapsed = time.time() - start

    rrf_results[query] = fused[:10]  # top 10
    rrf_times[query] = elapsed
    print(f"  [{elapsed:.2f}s] {query[:60]}")

print(f"\nAverage latency: {np.mean(list(rrf_times.values())):.3f}s")

---
## Strategy 3: Cross-Encoder Reranking

**How it works:** A cross-encoder takes a (query, passage) pair as input and outputs a single relevance score. Unlike bi-encoders (which embed query and passage independently), cross-encoders apply full cross-attention between the query and passage tokens, allowing much richer relevance modeling.

We use `cross-encoder/ms-marco-MiniLM-L-6-v2` — a small (22M params, ~80MB) model trained on MS MARCO passage ranking data.

**Workflow:**
1. Get top-20 hybrid results from the search index
2. Score each (query, passage) pair with the cross-encoder
3. Re-sort by cross-encoder score
4. Return top 10

**Pros:**
- High quality — joint query-passage attention captures nuanced relevance
- Runs locally — no API cost, no rate limits
- Well-established approach in the IR community

**Cons:**
- Requires model download (~80MB on first use)
- Adds ~100-500ms latency (depends on number of passages and hardware)
- Model trained on general web data — may not perfectly fit AWS technical documentation

In [ ]:
from sentence_transformers import CrossEncoder

# Load cross-encoder (downloads ~80MB on first run)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder model loaded.")

In [ ]:
def cross_encoder_rerank(
    query: str,
    hits: list[dict],
    model: CrossEncoder,
    top_k: int = 10,
) -> list[dict]:
    """Rerank search hits using a cross-encoder model.

    Args:
        query: The search query.
        hits: List of search hit dicts with 'content' field.
        model: A loaded CrossEncoder instance.
        top_k: Number of results to return after reranking.

    Returns:
        Reranked list of hit dicts with added 'ce_score' field.
    """
    if not hits:
        return []

    pairs = [(query, hit["content"]) for hit in hits]
    scores = model.predict(pairs)

    for hit, score in zip(hits, scores):
        hit["ce_score"] = round(float(score), 4)

    reranked = sorted(hits, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]


print("Cross-encoder rerank function defined.")

In [ ]:
# Cross-encoder: get top-20 hybrid, rerank to top-10
ce_results = {}
ce_times = {}

for query in TEST_QUERIES:
    start = time.time()
    resp = search(query, search_type="hybrid", size=20)
    reranked = cross_encoder_rerank(query, resp["hits"], cross_encoder, top_k=10)
    elapsed = time.time() - start

    ce_results[query] = reranked
    ce_times[query] = elapsed
    print(f"  [{elapsed:.2f}s] {query[:60]}")

print(f"\nAverage latency: {np.mean(list(ce_times.values())):.3f}s")

---
## Strategy 4: LLM-Based Reranking (Bedrock Claude 3 Haiku)

**How it works:** For each (query, passage) pair, ask an LLM to rate the passage's relevance on a 0-10 scale. Then re-sort by LLM scores.

We use **Claude 3 Haiku** via Bedrock — the smallest/cheapest Claude model — to minimize cost and latency.

**Workflow:**
1. Get top-10 hybrid results from the search index
2. For each result, call Bedrock Claude 3 Haiku with a relevance scoring prompt
3. Parse the numeric score and re-sort

**Pros:**
- Highest potential quality — LLMs understand nuance, context, and intent
- Can handle domain-specific terminology well
- No local model download required

**Cons:**
- **Cost**: ~$0.01 per full run (6 queries x 10 results = 60 API calls) — scales linearly
- **Latency**: ~1-2s per API call, so ~10-20s per query
- **Rate limits**: May hit Bedrock throttling with many concurrent calls
- Score parsing can be fragile (LLM may not always return just a number)

> **Cost warning**: Running this cell makes ~60 Bedrock API calls. Total cost is minimal (~$0.01) but be aware of rate limits.

In [ ]:
import boto3
import re

bedrock = boto3.client("bedrock-runtime", region_name="us-west-2")


def llm_rerank(
    query: str,
    hits: list[dict],
    client: boto3.client,
    top_k: int = 10,
) -> list[dict]:
    """Rerank search hits using Bedrock Claude 3 Haiku for relevance scoring.

    Args:
        query: The search query.
        hits: List of search hit dicts with 'content' field.
        client: A boto3 bedrock-runtime client.
        top_k: Number of results to return after reranking.

    Returns:
        Reranked list of hit dicts with added 'llm_score' field.
    """
    if not hits:
        return []

    for hit in hits:
        prompt = (
            f"Rate the relevance of this passage to the query on a scale of 0 to 10, "
            f"where 0 means completely irrelevant and 10 means perfectly relevant.\n\n"
            f"Query: {query}\n\n"
            f"Passage: {hit['content'][:500]}\n\n"
            f"Return ONLY a single number between 0 and 10. No explanation."
        )

        body = json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 10,
            "messages": [{"role": "user", "content": prompt}],
        })

        response = client.invoke_model(
            modelId="anthropic.claude-3-haiku-20240307-v1:0",
            body=body,
        )
        result = json.loads(response["body"].read())
        score_text = result["content"][0]["text"].strip()

        # Parse the numeric score, handling potential non-numeric responses
        match = re.search(r"\d+\.?\d*", score_text)
        score = float(match.group()) if match else 5.0
        hit["llm_score"] = min(score, 10.0)  # cap at 10

    reranked = sorted(hits, key=lambda x: x["llm_score"], reverse=True)
    return reranked[:top_k]


print("LLM rerank function defined.")

In [ ]:
# LLM reranking: get top-10 hybrid, score each with Claude 3 Haiku
llm_results = {}
llm_times = {}

for query in TEST_QUERIES:
    start = time.time()
    resp = search(query, search_type="hybrid", size=10)
    # Deep copy hits to avoid mutating baseline results
    hits_copy = [dict(h) for h in resp["hits"]]
    reranked = llm_rerank(query, hits_copy, bedrock, top_k=10)
    elapsed = time.time() - start

    llm_results[query] = reranked
    llm_times[query] = elapsed
    print(f"  [{elapsed:.2f}s] {query[:60]}")

print(f"\nAverage latency: {np.mean(list(llm_times.values())):.3f}s")

---
## Comparison: Side-by-Side Results

For each query, compare the top-5 results from each strategy. Look for rank changes — documents that move up or down significantly between strategies.

In [ ]:
STRATEGY_NAMES = ["Baseline", "RRF", "Cross-Encoder", "LLM"]


def get_strategy_results(query: str) -> dict[str, list[dict]]:
    """Get top results from all strategies for a query."""
    return {
        "Baseline": baseline_results[query]["hits"][:5],
        "RRF": rrf_results[query][:5],
        "Cross-Encoder": ce_results[query][:5],
        "LLM": llm_results[query][:5],
    }


for query in TEST_QUERIES:
    print(f"\n{'='*90}")
    print(f"Query: {query}")
    print(f"{'='*90}")

    strategy_hits = get_strategy_results(query)
    for name, hits in strategy_hits.items():
        print(f"\n--- {name} ---")
        rows = []
        for i, hit in enumerate(hits, start=1):
            score_col = hit.get("score", hit.get("rrf_score", hit.get("ce_score", hit.get("llm_score", 0))))
            rows.append({
                "rank": i,
                "filename": hit["filename"],
                "score": round(score_col, 4) if isinstance(score_col, float) else score_col,
                "content_preview": hit["content"][:80],
            })
        display(pd.DataFrame(rows))

## Rank Correlation Analysis

How much do the strategies agree on ranking? We compute Kendall's tau rank correlation coefficient between each pair of strategies, averaged across all queries.

- tau = 1.0 means perfect agreement
- tau = 0.0 means no correlation
- tau = -1.0 means perfect disagreement

In [ ]:
from scipy.stats import kendalltau


def compute_rank_correlation(hits_a: list[dict], hits_b: list[dict]) -> float:
    """Compute Kendall's tau between two ranked result lists.

    Only considers documents that appear in both lists.
    """
    ids_a = [h["doc_id"] for h in hits_a]
    ids_b = [h["doc_id"] for h in hits_b]
    common = set(ids_a) & set(ids_b)

    if len(common) < 2:
        return 0.0

    ranks_a = {doc_id: rank for rank, doc_id in enumerate(ids_a) if doc_id in common}
    ranks_b = {doc_id: rank for rank, doc_id in enumerate(ids_b) if doc_id in common}

    common_sorted = sorted(common)
    ra = [ranks_a[d] for d in common_sorted]
    rb = [ranks_b[d] for d in common_sorted]

    tau, _ = kendalltau(ra, rb)
    return round(tau, 3) if not np.isnan(tau) else 0.0


# Compute average rank correlation between all strategy pairs
correlation_data = {}
for i, name_a in enumerate(STRATEGY_NAMES):
    for name_b in STRATEGY_NAMES[i + 1:]:
        taus = []
        for query in TEST_QUERIES:
            hits = get_strategy_results(query)
            tau = compute_rank_correlation(hits[name_a], hits[name_b])
            taus.append(tau)
        correlation_data[f"{name_a} vs {name_b}"] = round(np.mean(taus), 3)

print("Average Kendall's Tau (rank correlation):")
for pair, tau in correlation_data.items():
    print(f"  {pair}: {tau}")

## Latency Comparison

How much time does each strategy add to the search pipeline?

In [ ]:
latency_summary = pd.DataFrame({
    "Strategy": STRATEGY_NAMES,
    "Avg Latency (s)": [
        round(np.mean(list(baseline_times.values())), 3),
        round(np.mean(list(rrf_times.values())), 3),
        round(np.mean(list(ce_times.values())), 3),
        round(np.mean(list(llm_times.values())), 3),
    ],
    "Min (s)": [
        round(min(baseline_times.values()), 3),
        round(min(rrf_times.values()), 3),
        round(min(ce_times.values()), 3),
        round(min(llm_times.values()), 3),
    ],
    "Max (s)": [
        round(max(baseline_times.values()), 3),
        round(max(rrf_times.values()), 3),
        round(max(ce_times.values()), 3),
        round(max(llm_times.values()), 3),
    ],
})
display(latency_summary)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(STRATEGY_NAMES, latency_summary["Avg Latency (s)"], color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
ax.set_ylabel("Average Latency (seconds)")
ax.set_title("Reranking Strategy Latency Comparison")
for i, v in enumerate(latency_summary["Avg Latency (s)"]):
    ax.text(i, v + 0.01, f"{v:.3f}s", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## Strategy Summary

| Strategy | Quality | Latency | Cost | Complexity | Best For |
|----------|---------|---------|------|------------|----------|
| **No reranking** | Moderate | Lowest (~1 API call) | Free | None | Default production use |
| **RRF** | Moderate+ | Low (~2 API calls) | Free | Low (pure algorithm) | Simple improvement over weighted avg |
| **Cross-encoder** | High | Medium (~100-500ms local) | Free (local model) | Medium (model download) | Quality-focused with acceptable latency |
| **LLM (Claude)** | Highest potential | High (~10-20s per query) | ~$0.01/run | Medium (API calls) | Offline evaluation, quality upper bound |

## Observations and Recommendations

_Fill in after running the notebook with actual results._

### Questions to Answer

1. **Did any reranking strategy consistently improve over the baseline?**
   - _TODO_

2. **How much rank correlation exists between strategies?**
   - _TODO_

3. **Is the latency cost of cross-encoder reranking acceptable for production?**
   - _TODO_

4. **Did the LLM reranker agree with the cross-encoder, or did they diverge?**
   - _TODO (if they agree, the cheaper cross-encoder may be sufficient)_

5. **Recommendation for production:**
   - _TODO: Which strategy offers the best quality/latency/cost tradeoff?_
   - _TODO: Should reranking be implemented as a configurable option?_